In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
df=pd.read_csv(os.path.join(path, "Q3_data.csv"))

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")
check_missing_values(df)

In [ ]:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 50].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_dat=missing_data['Column'].to_numpy()

In [ ]:
# i will drop columns that have nan value grater than 50
df_clean=df.drop(columns=missing_dat)
df_clean

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df_clean.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 20].sort_values('Missing_Percentage', ascending=False)
missing_dtat2=missing_data['Column'].to_numpy()

In [ ]:
a=missing_data['Column'].to_numpy()

In [ ]:
df_clean[missing_dtat2] = df_clean[missing_dtat2].fillna(df_clean[missing_dtat2].mean()[0])
df_clean=df_clean.fillna(0)
df_clean=df_clean.drop(a,axis=1)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
df_clean.columns

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop('Target')  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df_clean.head()

In [ ]:
# Task 5: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
F1=[]
Acc=[]
from sklearn.model_selection import StratifiedKFold #because target is imbalance
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

n_splits=5
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  model=CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  f1=f1_score(y_test, y_pred)
  acc=accuracy_score
  F1.append(f1)
  Acc.append(acc)


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

importances = model.feature_importances_
feature_cols=X.columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
np.max(model.feature_importances_)

In [ ]:
# Task Bonus: Write your code here: